# Program synthesis — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sigmoid(z): return 1/(1+np.exp(-z))
def bce(p,y):
    p=np.clip(np.asarray(p,dtype=float),1e-9,1-1e-9)
    y=np.asarray(y,dtype=float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))
print('setup ok')

## Searching a small program space for code that matches examples
Program synthesis turns input-output examples into a search problem over candidate programs. These five examples enumerate tiny arithmetic programs, visualize their losses, show how each example prunes the version space, expose ambiguity from too few examples, and recover the intended program end-to-end.

In [ ]:
# 1) enumerate a tiny arithmetic language and score each program on examples
xs=np.array([0,-2,-1,1,2],dtype=float); ys=xs**2+1
programs=[]
for c in [-2,-1,0,1,2]:
    programs.append((f'x+{c}',lambda x,c=c: x+c)); programs.append((f'{c}*x',lambda x,c=c: c*x)); programs.append((f'x^2+{c}',lambda x,c=c: x*x+c))
losses=np.array([np.mean((f(xs)-ys)**2) for _,f in programs]); best=int(np.argmin(losses))
plt.figure(figsize=(7,3)); plt.bar(range(len(programs)),losses); plt.xticks(range(len(programs)),[n for n,_ in programs],rotation=70); plt.ylabel('MSE'); plt.title(f'best program: {programs[best][0]}')
assert programs[best][0]=='x^2+1' and abs(float(losses[best]))<1e-12

In [ ]:
# 2) each input-output example prunes inconsistent programs
counts=[]; labels=[]
for m in range(1,len(xs)+1):
    ok=sum(np.allclose(f(xs[:m]),ys[:m]) for _,f in programs); counts.append(ok); labels.append(str(m))
plt.figure(figsize=(6,3)); plt.plot(range(1,6),counts,'-o'); plt.xticks(range(1,6),labels); plt.xlabel('examples shown'); plt.ylabel('consistent programs'); plt.title('version space shrinks as examples arrive')
assert counts==[2,1,1,1,1]

In [ ]:
# 3) with one example, ambiguity is real: multiple programs fit x=0 -> 1
fits=[n for n,f in programs if np.allclose(f(xs[:1]),ys[:1])]
plt.figure(figsize=(6,3)); plt.bar(fits,[1]*len(fits),color='tab:orange'); plt.ylabel('fits first example'); plt.title('too few examples leave aliases')
assert set(fits)=={'x+1','x^2+1'}

In [ ]:
# 4) search budget matters: rank programs by loss to see near misses
order=np.argsort(losses); top_names=[programs[i][0] for i in order[:5]]; top_losses=losses[order[:5]]
plt.figure(figsize=(6,3)); plt.bar(top_names,top_losses,color=['tab:green']+['tab:gray']*4); plt.ylabel('MSE'); plt.title('top of the synthesis search')
assert top_names[:3]==['x^2+1','x^2+0','x^2+2']

In [ ]:
# 5) the synthesized program generalizes on held-out inputs
xheld=np.array([3.,4.]); yheld=xheld**2+1; fbest=programs[best][1]; pred=fbest(xheld)
plt.figure(figsize=(6,3)); plt.plot(xheld,yheld,'ko',label='target'); plt.plot(xheld,pred,'rx',ms=10,label='synthesized'); plt.legend(); plt.title('exact program matches held-out cases')
assert np.allclose(pred,np.array([10.,17.]))